# AdeSEG utility-aware memory retrieval

LEGACY / DO NOT RUN FOR THE V2 LATENT-UTILITY CLAIM. This notebook uses the earlier utility-retrieval v1 protocol, a smaller checkpoint, and probe-correlation gates that do not establish the required E4 or E5 evidence. See `protocols/latent_utility_v2.json` and `docs/phase_a_implementation_audit_2026-10-17.md`.

In [ ]:
raise RuntimeError('Legacy v1 notebook cannot establish latent-utility v2 claims; use a validated v2 runner')
import importlib.util, json, os, shutil, subprocess, sys, time
from pathlib import Path
from kaggle_secrets import UserSecretsClient

WORK = Path('/kaggle/working')
REPO = WORK / 'AdeSEG'
OUT = WORK / 'outputs' / '16:10:2026'
OUT.mkdir(parents=True, exist_ok=True)
started = time.time()

def mark(stage, state='running', **details):
    payload = {'stage': stage, 'state': state, 'elapsed_seconds': time.time() - started, **details}
    status_dir = OUT/'status'; status_dir.mkdir(parents=True, exist_ok=True)
    temporary = status_dir/'orchestrator.json.tmp'
    temporary.write_text(json.dumps(payload, indent=2)+'\n')
    os.replace(temporary, status_dir/'orchestrator.json')
    with (status_dir/'events.jsonl').open('a') as stream:
        stream.write(json.dumps(payload, sort_keys=True)+'\n')
    print('[status]', json.dumps(payload, sort_keys=True), flush=True)

def run(command, cwd=None, env=None, timeout=7200):
    command = list(map(str, command))
    mark('command', command=' '.join(command))
    print('+', ' '.join(command), flush=True)
    command_started = time.time()
    try:
        subprocess.run(command, cwd=cwd, env=env or os.environ.copy(), check=True, timeout=timeout)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        mark('command_failed', state='failed', command=' '.join(command), error=repr(exc))
        raise
    print(f'[done in {time.time() - command_started:.1f}s]', ' '.join(command), flush=True)

mark('bootstrap')
wandb_api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
assert wandb_api_key, 'Kaggle secret WANDB_API_KEY is empty'
os.environ.update({
    'WANDB_API_KEY': wandb_api_key,
    'WANDB_BASE_URL': 'https://forge.coreweave.com/api/wandb',
    'WANDB_PROJECT': 'ade-seg',
    'WANDB_ENTITY': 'adenoid-hypertrophy',
    'PYTHONUNBUFFERED': '1',
    'GIT_TERMINAL_PROMPT': '0',
})
mark('secret_loaded')
if REPO.exists():
    shutil.rmtree(REPO)
run(['git', 'clone', '--depth', '1', '--branch', 'experiment/utility-aware-memory-2026-10-16', 'https://github.com/aCoderChild/AdeSEG.git', REPO], timeout=300)
if importlib.util.find_spec('wandb') is None:
    run([sys.executable, '-m', 'pip', 'install', 'wandb'], timeout=300)
run([sys.executable, REPO/'scripts/verify_wandb_connection.py', '--name', 'kaggle-full-connectivity-2026-10-16'], cwd=REPO, timeout=120)
mark('wandb_connected')
run(['git', 'submodule', 'update', '--init', '--depth', '1', 'external/MedSAM2', 'external/sam2', 'external/SAM2Long', 'external/Lean-SAM2'], cwd=REPO, timeout=900)
torch_before = __import__('torch').__version__
missing = [package for module, package in {'hydra': 'hydra-core>=1.3.2', 'iopath': 'iopath>=0.1.10'}.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, '-m', 'pip', 'install', *missing], timeout=300)
if importlib.util.find_spec('tensordict') is None:
    run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'tensordict>=0.7,<0.8'], timeout=300)
import cv2, scipy, tensordict, torch
assert torch.__version__ == torch_before and torch.cuda.is_available(), (torch_before, torch.__version__, torch.cuda.is_available())
BASE_ENV = os.environ.copy()
mark('repository_ready', torch=torch.__version__, gpu=torch.cuda.get_device_name(0))

In [ ]:
PROTOCOL = REPO / 'protocols/utility_retrieval_v1.json'
INPUT_ROOT = Path('/kaggle/input')
DAVIS_ROOT = INPUT_ROOT
POLYP_ROOT = INPUT_ROOT
DATA = WORK / 'experiment_data'
DATA.mkdir(exist_ok=True)
DAVIS_MANIFEST = DATA / 'davis.jsonl'
POLYP_MANIFEST = DATA / 'polypgen.jsonl'
run([sys.executable, REPO/'scripts/build_davis_manifest.py', '--data-root', DAVIS_ROOT, '--protocol', PROTOCOL, '--output', DAVIS_MANIFEST], cwd=REPO)
run([sys.executable, REPO/'scripts/build_polypgen_kaggle_manifest.py', '--data-root', POLYP_ROOT, '--output', POLYP_MANIFEST], cwd=REPO)
CKPT = DATA / 'sam2.1_hiera_small.pt'
if not CKPT.exists():
    run(['wget', '--show-progress', '--timeout=30', '--tries=3', 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt', '-O', CKPT], timeout=300)
assert CKPT.stat().st_size > 100_000_000, CKPT.stat().st_size
CFG = 'configs/sam2.1_hiera_s1024.yaml'
lean_ckpt = REPO/'external/Lean-SAM2/checkpoints/sam2.1_hiera_small.pt'
if not lean_ckpt.exists():
    lean_ckpt.symlink_to(CKPT)
davis_candidates = sorted({path.parent.parent for path in INPUT_ROOT.glob('**/JPEGImages/480p') if (path.parent.parent/'Annotations/480p').is_dir()})
assert len(davis_candidates) == 1, davis_candidates
davis = davis_candidates[0]
mark('data_and_checkpoint_ready', checkpoint_bytes=CKPT.stat().st_size)
print('Manifests and fixed SAM2.1 Hiera-S checkpoint ready; DAVIS root:', davis)

In [ ]:
# Sanity: official SAM2.1 versus the instrumented fork, same checkpoint/protocol.
mark('sanity_official_sam21')
official_masks = OUT/'sanity/official_sam21_masks'
instrumented = OUT/'sanity/instrumented_sam21'
run([sys.executable, REPO/'external/sam2/tools/vos_inference.py', '--sam2_cfg', 'configs/sam2.1/sam2.1_hiera_s.yaml', '--sam2_checkpoint', CKPT, '--base_video_dir', davis/'JPEGImages/480p', '--input_mask_dir', davis/'Annotations/480p', '--video_list_file', davis/'ImageSets/2017/val.txt', '--output_mask_dir', official_masks], cwd=REPO/'external/sam2', timeout=7200)
mark('sanity_instrumented_sam21')
run([sys.executable, REPO/'scripts/infer.py', '--sam2_cfg', CFG, '--sam2_checkpoint', CKPT, '--manifest', DAVIS_MANIFEST, '--split', 'val', '--label_ids', *map(str, range(1, 11)), '--prompt_type', 'mask', '--device', 'cuda', '--output_dir', instrumented], cwd=REPO, env=BASE_ENV)
official_wrapper = OUT/'sanity/official_wrapper'
official_wrapper.mkdir(parents=True, exist_ok=True)
(official_wrapper/'masks').symlink_to(official_masks)
run([sys.executable, REPO/'scripts/evaluate_davis.py', '--manifest', DAVIS_MANIFEST, '--split', 'val', '--pred-dir', official_wrapper, '--output', OUT/'sanity/official.json'], cwd=REPO)
run([sys.executable, REPO/'scripts/evaluate_davis.py', '--manifest', DAVIS_MANIFEST, '--split', 'val', '--pred-dir', instrumented, '--output', OUT/'sanity/instrumented.json'], cwd=REPO)
official = json.loads((OUT/'sanity/official.json').read_text())['summary']['J&F']
observed = json.loads((OUT/'sanity/instrumented.json').read_text())['summary']['J&F']
sanity = {'official_J&F': official, 'instrumented_J&F': observed, 'absolute_point_difference': 100*abs(official-observed), 'pass': 100*abs(official-observed) <= 1.0}
(OUT/'sanity/check.json').write_text(json.dumps(sanity, indent=2)+'\n')
assert sanity['pass'], sanity
mark('sanity_complete', passed=sanity['pass'], point_difference=sanity['absolute_point_difference'])
sanity

In [ ]:
def utility_run(name, split='val', memory_source='predicted', corruption_level=0.0, frame_selection='failure_onsets', random_null=True):
    target = OUT/name
    command = [sys.executable, REPO/'scripts/generate_utility_labels.py', '--sam2-cfg', CFG, '--sam2-checkpoint', CKPT, '--manifest', DAVIS_MANIFEST, '--split', split, '--protocol', PROTOCOL, '--output-dir', target, '--device', 'cuda', '--frame-selection', frame_selection, '--memory-source', memory_source, '--corruption-level', str(corruption_level), '--wandb-base-url', os.environ['WANDB_BASE_URL'], '--wandb-name', name]
    command.append('--random-null' if random_null else '--no-random-null')
    run(command, cwd=REPO, env=BASE_ENV)
    return target

predicted = utility_run('E1_E2_E3/predicted')
predicted_summary = json.loads((predicted/'summary.json').read_text())
assert predicted_summary['sanity_no_op_max_abs'] == 0.0, predicted_summary
assert predicted_summary['sanity_cached_state_restore_max_abs'] == 0.0, predicted_summary
assert predicted_summary['sanity_attention_reconstruction_max_abs'] is not None, predicted_summary
assert predicted_summary['sanity_attention_reconstruction_max_abs'] < 1e-5, predicted_summary
mark('utility_instrumentation_sanity_complete')
ground_truth = utility_run('E1_E2_E3/ground_truth', memory_source='ground_truth')
corruptions = []
for level in (0.25, 0.50, 0.75):
    corruptions.append((level, utility_run(f'E1_E2_E3/corruption_{level:.2f}', memory_source='corrupted', corruption_level=level, random_null=False)))
run([sys.executable, REPO/'scripts/classify_failure_taxonomy.py', '--predicted', predicted/'frames.jsonl', '--ground-truth-memory', ground_truth/'frames.jsonl', '--output', OUT/'E1_E2_E3/taxonomy.json'], cwd=REPO)
mechanism_cmd = [sys.executable, REPO/'scripts/analyze_utility_mechanism.py', '--condition', 'predicted', predicted/'frames.jsonl', '--condition', 'ground_truth', ground_truth/'frames.jsonl']
for level, path in corruptions:
    mechanism_cmd += ['--condition', f'corruption_{level:.2f}', path/'frames.jsonl']
mechanism_cmd += ['--output', OUT/'E1_E2_E3/mechanism.json']
run(mechanism_cmd, cwd=REPO)
taxonomy = json.loads((OUT/'E1_E2_E3/taxonomy.json').read_text())['summary']
mechanism = json.loads((OUT/'E1_E2_E3/mechanism.json').read_text())['summary']
go_no_go = {'E2_pass': taxonomy['mean_recovered_fraction'] is not None and taxonomy['mean_recovered_fraction'] >= 0.33, 'E3_pass': mechanism['E3_pass']}
(OUT/'E1_E2_E3/go_no_go.json').write_text(json.dumps(go_no_go, indent=2)+'\n')
go_no_go

In [ ]:
# E5 label generation and three-seed probe. Test data is not used for selection.
mark('E5_label_generation')
train_labels = utility_run('E5/train_labels', split='train', frame_selection='all', random_null=False)
holdout_labels = utility_run('E5/holdout_labels', split='holdout', frame_selection='all', random_null=False)
probe_summaries = []
for seed in (0, 1, 2):
    probe = OUT/f'E5/seed_{seed}'
    run([sys.executable, REPO/'scripts/train_utility_probe.py', '--train-records', train_labels/'utility_records.pt', '--evaluation-records', holdout_labels/'utility_records.pt', '--output-dir', probe, '--seed', seed, '--device', 'cuda', '--wandb-base-url', os.environ['WANDB_BASE_URL'], '--wandb-name', f'E5-probe-seed-{seed}'], cwd=REPO, env=BASE_ENV)
    probe_summaries.append(json.loads((probe/'summary.json').read_text()))
E5_pass = sum(item['E5_pass'] for item in probe_summaries) >= 2
decision = {'E2_pass': go_no_go['E2_pass'], 'E3_pass': go_no_go['E3_pass'], 'E5_pass': E5_pass, 'E6_authorized_by_gate': bool(E5_pass)}
(OUT/'decision.json').write_text(json.dumps(decision, indent=2)+'\n')
decision

In [ ]:
# E6/E7/E8 only after the preregistered learnability gate.
if E5_pass:
    for seed in (0, 1, 2):
        baseline = OUT/f'E6_E8/attention_baseline_seed_{seed}'
        run([sys.executable, REPO/'scripts/train_attention_baseline.py', '--sam2-cfg', CFG, '--sam2-checkpoint', CKPT, '--manifest', DAVIS_MANIFEST, '--train-split', 'train', '--validation-split', 'holdout', '--output-dir', baseline, '--seed', seed, '--device', 'cuda', '--wandb-base-url', os.environ['WANDB_BASE_URL'], '--wandb-name', f'attention-baseline-seed-{seed}'], cwd=REPO, env=BASE_ENV)
    for seed in (0, 1, 2):
        for dataset, manifest, split, labels, evaluator in ((
            'davis', DAVIS_MANIFEST, 'val', list(range(1, 11)), 'davis'),
            ('polypgen', POLYP_MANIFEST, 'test', [1], 'binary')
        ):
            result = OUT/f'E6_E8/ours_seed_{seed}_{dataset}'
            run([sys.executable, REPO/'scripts/infer.py', '--sam2_cfg', CFG, '--sam2_checkpoint', CKPT, '--manifest', manifest, '--split', split, '--label_ids', *map(str, labels), '--prompt_type', 'mask', '--utility_checkpoint', OUT/f'E5/seed_{seed}/utility_head.pt', '--device', 'cuda', '--output_dir', result], cwd=REPO)
            script = 'evaluate_davis.py' if evaluator == 'davis' else 'evaluate_binary_manifest.py'
            run([sys.executable, REPO/f'scripts/{script}', '--manifest', manifest, '--split', split, '--pred-dir', result, '--output', result/'evaluation.json'], cwd=REPO)
            baseline_result = OUT/f'E6_E8/attention_seed_{seed}_{dataset}'
            run([sys.executable, REPO/'scripts/infer.py', '--sam2_cfg', CFG, '--sam2_checkpoint', CKPT, '--manifest', manifest, '--split', split, '--label_ids', *map(str, labels), '--prompt_type', 'mask', '--utility_checkpoint', OUT/f'E6_E8/attention_baseline_seed_{seed}/attention_baseline.pt', '--device', 'cuda', '--output_dir', baseline_result], cwd=REPO)
            run([sys.executable, REPO/f'scripts/{script}', '--manifest', manifest, '--split', split, '--pred-dir', baseline_result, '--output', baseline_result/'evaluation.json'], cwd=REPO)
    native_polyp = OUT/'E6_E8/native_polypgen'
    run([sys.executable, REPO/'scripts/infer.py', '--sam2_cfg', CFG, '--sam2_checkpoint', CKPT, '--manifest', POLYP_MANIFEST, '--split', 'test', '--label_ids', '1', '--prompt_type', 'mask', '--device', 'cuda', '--output_dir', native_polyp], cwd=REPO)
    run([sys.executable, REPO/'scripts/evaluate_binary_manifest.py', '--manifest', POLYP_MANIFEST, '--split', 'test', '--pred-dir', native_polyp, '--output', native_polyp/'evaluation.json'], cwd=REPO)
else:
    print('E6/E7/E8 utility-head runs skipped exactly as preregistered because E5 failed.')

In [ ]:
# Strong upstream baselines on DAVIS, using the same Hiera-S checkpoint and first-frame masks.
if E5_pass:
    sam2long_masks = OUT/'E6_E8/sam2long_davis_masks'
    run([sys.executable, REPO/'external/SAM2Long/tools/vos_inference.py', '--sam2_cfg', 'configs/sam2.1/sam2.1_hiera_s.yaml', '--sam2_checkpoint', CKPT, '--base_video_dir', davis/'JPEGImages/480p', '--input_mask_dir', davis/'Annotations/480p', '--video_list_file', davis/'ImageSets/2017/val.txt', '--output_mask_dir', sam2long_masks, '--num_pathway', '3', '--iou_thre', '0.1', '--uncertainty', '2'], cwd=REPO/'external/SAM2Long')
    wrapper = OUT/'E6_E8/sam2long_wrapper'; wrapper.mkdir(parents=True, exist_ok=True); (wrapper/'masks').symlink_to(sam2long_masks)
    run([sys.executable, REPO/'scripts/evaluate_davis.py', '--manifest', DAVIS_MANIFEST, '--split', 'val', '--pred-dir', wrapper, '--output', OUT/'E6_E8/sam2long_davis.json'], cwd=REPO)
    lean_masks = OUT/'E6_E8/lean_sam2_davis_masks'
    run([sys.executable, REPO/'external/Lean-SAM2/tools/vos_inference.py', '--sam2_model', 'small', '--base_video_dir', davis/'JPEGImages/480p', '--input_mask_dir', davis/'Annotations/480p', '--video_list_file', davis/'ImageSets/2017/val.txt', '--output_mask_dir', lean_masks], cwd=REPO/'external/Lean-SAM2')
    wrapper = OUT/'E6_E8/lean_wrapper'; wrapper.mkdir(parents=True, exist_ok=True); (wrapper/'masks').symlink_to(lean_masks)
    run([sys.executable, REPO/'scripts/evaluate_davis.py', '--manifest', DAVIS_MANIFEST, '--split', 'val', '--pred-dir', wrapper, '--output', OUT/'E6_E8/lean_sam2_davis.json'], cwd=REPO)

In [ ]:
# Compact W&B artifact: protocol, decisions, summaries and learned heads (not mask volumes).
import wandb
final = wandb.init(project='ade-seg', entity='adenoid-hypertrophy', name='utility-retrieval-final-2026-10-16', tags=['utility', 'preregistered', 'kaggle'])
final.config.update({'protocol': str(PROTOCOL), **decision})
final.log({f'decision/{key}': int(value) for key, value in decision.items()})
artifact = wandb.Artifact('utility-retrieval-2026-10-16', type='experiment-results')
artifact.add_file(str(PROTOCOL), name='protocol.json')
for path in OUT.rglob('*.json'):
    artifact.add_file(str(path), name=str(path.relative_to(OUT)))
for path in OUT.glob('E5/seed_*/utility_head.pt'):
    artifact.add_file(str(path), name=str(path.relative_to(OUT)))
final.log_artifact(artifact)
final.finish()
mark('complete', state='complete', **decision)
print('Completed. Kaggle output root:', OUT)